# RA-Trabecular — Notebook 01: First reproducible example

**Version:** 0.1.0  
**Author:** Verónica Zumpano Blumenfeld ([ORCID 0009-0006-2030-1849](https://orcid.org/0009-0006-2030-1849))  
**License:** Apache 2.0

This notebook demonstrates the full RA-Trabecular pipeline end-to-end:

1. Generate a 3D Voronoi-based trabecular network.
2. Apply progressive erosion under each of the three available regimes.
3. Compute the IRCE trajectory and its individual components.
4. Visualize the predicted critical transition.

The notebook is intentionally minimal and self-contained so that it serves as the canonical reproducible artifact for v0.1.0.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from ra_trabecular import (
    generate_voronoi_network,
    progressive_erosion,
    ErosionRegime,
    compute_irce,
    compute_irce_multiplicative,
)

RNG_SEED = 42

## 1. Generate the pristine network

We generate a 3D Voronoi tessellation with mild anisotropy along the z-axis (mimicking principal stress alignment of cancellous bone).

In [ ]:
pristine = generate_voronoi_network(
    n_seeds=200,
    domain_size=10.0,
    anisotropy_factor=1.4,
    rng_seed=RNG_SEED,
)

print(f'Nodes:               {pristine.n_nodes}')
print(f'Edges:               {pristine.n_edges}')
print(f'Mean coordination z: {pristine.mean_coordination:.3f}')

## 2. Generate erosion trajectories under three regimes

We simulate progressive erosion up to 60% edge removal in 25 steps under each regime.

In [ ]:
regimes = [ErosionRegime.UNIFORM, ErosionRegime.PERIPHERAL, ErosionRegime.OSTEOCLASTIC]
trajectories = {
    r: progressive_erosion(
        pristine, n_steps=25, max_removal_fraction=0.60,
        regime=r, rng_seed=RNG_SEED
    )
    for r in regimes
}
print('Trajectories generated for:', [r.value for r in regimes])

## 3. Compute IRCE along each trajectory

In [ ]:
def evaluate_trajectory(traj, ref):
    p_rem, irce_add, mod_r, conn_r, aniso, irce_mul = [], [], [], [], [], []
    for state in traj:
        r_add = compute_irce(state.network, ref)
        r_mul = compute_irce_multiplicative(state.network, ref)
        p_rem.append(state.p_removed)
        irce_add.append(r_add.irce)
        mod_r.append(r_add.modulus_ratio)
        conn_r.append(r_add.connectivity_ratio)
        aniso.append(r_add.anisotropy)
        irce_mul.append(r_mul.irce)
    return {
        'p_rem': np.array(p_rem),
        'irce_additive': np.array(irce_add),
        'irce_multiplicative': np.array(irce_mul),
        'modulus_ratio': np.array(mod_r),
        'connectivity_ratio': np.array(conn_r),
        'anisotropy': np.array(aniso),
    }

results = {r: evaluate_trajectory(traj, pristine) for r, traj in trajectories.items()}
print('IRCE trajectories computed.')

## 4. Visualize the IRCE trajectory and its components

The four panels below display:

- Top-left: IRCE (additive form) vs. fraction of removed edges, for the three regimes.
- Top-right: IRCE (multiplicative form) vs. fraction of removed edges.
- Bottom-left: individual IRCE components for the uniform regime.
- Bottom-right: comparison of additive vs. multiplicative formulations under the uniform regime.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
regime_colors = {
    ErosionRegime.UNIFORM: '#2E5599',
    ErosionRegime.PERIPHERAL: '#C0392B',
    ErosionRegime.OSTEOCLASTIC: '#27AE60',
}

ax = axes[0, 0]
for r, res in results.items():
    ax.plot(res['p_rem'], res['irce_additive'], 'o-', color=regime_colors[r], label=r.value, lw=2)
ax.set_xlabel('Fraction of removed edges $p_{rem}$')
ax.set_ylabel('IRCE (additive)')
ax.set_title('IRCE additive vs. erosion')
ax.grid(alpha=0.3); ax.legend()

ax = axes[0, 1]
for r, res in results.items():
    ax.plot(res['p_rem'], res['irce_multiplicative'], 's-', color=regime_colors[r], label=r.value, lw=2)
ax.set_xlabel('Fraction of removed edges $p_{rem}$')
ax.set_ylabel('IRCE (multiplicative)')
ax.set_title('IRCE multiplicative vs. erosion')
ax.grid(alpha=0.3); ax.legend()

ax = axes[1, 0]
res = results[ErosionRegime.UNIFORM]
ax.plot(res['p_rem'], res['modulus_ratio'], 'o-', label=r'$E^*_{eroded}/E^*_{healthy}$', color='#1F3864', lw=2)
ax.plot(res['p_rem'], res['connectivity_ratio'], 's-', label=r'$N_{LCC}/N_{total}$', color='#C0392B', lw=2)
ax.plot(res['p_rem'], res['anisotropy'], '^-', label=r'$A_{local}$', color='#27AE60', lw=2)
ax.set_xlabel('Fraction of removed edges $p_{rem}$')
ax.set_ylabel('Component value')
ax.set_title('IRCE components (uniform regime)')
ax.grid(alpha=0.3); ax.legend()

ax = axes[1, 1]
ax.plot(res['p_rem'], res['irce_additive'], 'o-', label='additive', color='#1F3864', lw=2)
ax.plot(res['p_rem'], res['irce_multiplicative'], 's-', label='multiplicative', color='#C0392B', lw=2)
ax.set_xlabel('Fraction of removed edges $p_{rem}$')
ax.set_ylabel('IRCE')
ax.set_title('Additive vs. multiplicative (uniform regime)')
ax.grid(alpha=0.3); ax.legend()

plt.tight_layout()
plt.savefig('../figures/01_irce_trajectory.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Locate the critical inflection point

We estimate $p_c$ as the location of maximal negative second derivative of IRCE along the trajectory.

In [ ]:
def estimate_critical_point(p_rem, irce):
    irce = np.asarray(irce); p_rem = np.asarray(p_rem)
    d1 = np.gradient(irce, p_rem)
    d2 = np.gradient(d1,   p_rem)
    idx = int(np.argmin(d2))
    return p_rem[idx], irce[idx]

print(f"{'regime':<15}{'p_c':>8}{'IRCE_c':>10}")
print('-' * 33)
for r, res in results.items():
    p_c, irce_c = estimate_critical_point(res['p_rem'], res['irce_additive'])
    print(f"{r.value:<15}{p_c:>8.3f}{irce_c:>10.4f}")

## 6. Interpretation

Observed features (qualitative; quantitative validation pending v0.4.0):

- IRCE decreases monotonically along all erosion regimes.
- Modulus ratio decays faster than connectivity ratio — consistent with rigidity percolation preceding geometric percolation in bending-dominated networks.
- Peripheral and osteoclastic regimes produce sharper transitions than uniform removal — biologically realistic, since clinical erosion is spatially clustered.
- Multiplicative IRCE drops more abruptly near the critical region — captures synergistic collapse better.

This notebook establishes the canonical reproducible artifact for v0.1.0. Subsequent releases will extend this analysis with:

- Multi-realization statistics (v0.2.0).
- Full FEM modulus computation (v0.3.0).
- Validation against public micro-CT datasets (v0.4.0).

---

*Cite this work via the DOI listed in `CITATION.cff` or the badge in `README.md`.*